# 06 — DAB-positive cell counting

`PositiveCellCounter` is marker-agnostic: it detects brown DAB signal using
an HSV color gate, applies a patch-specific Otsu threshold, filters
connected components by area, and reports count and density.

Install `.[cellcount]`. Validate segmentation on representative tissue
before treating counts as biological measurements.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path


def find_project_root(start: Path | None = None) -> Path:
    '''Find the RocqiPath repository whether Jupyter starts at root or how_to_use.'''
    here = (start or Path.cwd()).resolve()
    for candidate in (here, *here.parents):
        if (candidate / "pyproject.toml").is_file() and (
            candidate / "src" / "rocqipath"
        ).is_dir():
            return candidate
    raise FileNotFoundError(
        "RocqiPath repository not found. Start Jupyter inside the cloned repository."
    )


PROJECT_ROOT = find_project_root()
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

DATA_ROOT = PROJECT_ROOT / "data"
RESULTS_ROOT = PROJECT_ROOT / "results"

print(f"Project : {PROJECT_ROOT}")
print(f"Data    : {DATA_ROOT}")
print(f"Results : {RESULTS_ROOT}")


In [ ]:
IHC_SLIDE = DATA_ROOT / "wsi" / "Sample_0001_cd8.svs"
IHC_BATCH_DIR = DATA_ROOT / "wsi" / "cd8"
OUTPUT_ROOT = RESULTS_ROOT

TARGET_MAGNIFICATION = 20.0
SOURCE_MAGNIFICATION = None

RUN_SYNTHETIC_DEMO = False
RUN_SINGLE_SLIDE = False
RUN_BATCH = False
RUN_PAIRED_COMPARISON = False


## Create a small synthetic DAB slide

This is a software smoke test, not a biological validation. Brown circles
are drawn on a light tissue background so the full slide API can run on an
ordinary TIFF with an explicit 20x source magnification.


In [ ]:
import os

from PIL import Image, ImageDraw

demo_root = Path(
    os.environ.get(
        "ROCQIPATH_NOTEBOOK_DEMO_DIR",
        str(PROJECT_ROOT / "notebook_demo_outputs"),
    )
)
synthetic_slide = demo_root / "cell_counting" / "synthetic_cd8.tif"
synthetic_slide.parent.mkdir(parents=True, exist_ok=True)

image = Image.new("RGB", (512, 512), (220, 190, 205))
draw = ImageDraw.Draw(image)
for row in range(6):
    for col in range(8):
        x = 45 + col * 58
        y = 55 + row * 72
        draw.ellipse((x - 10, y - 10, x + 10, y + 10), fill=(180, 125, 70))
        draw.ellipse((x - 6, y - 6, x + 6, y + 6), fill=(85, 42, 18))
image.save(synthetic_slide, format="TIFF")
image.close()
print(synthetic_slide)


In [ ]:
from rocqipath.config import CellCountingConfig

count_cfg = CellCountingConfig(
    patch_size=512,
    tissue_threshold=0.10,
    target_magnification=TARGET_MAGNIFICATION,
    source_magnification=SOURCE_MAGNIFICATION,
    paired_source_magnification=None,
    output_dir=str(OUTPUT_ROOT),
    min_cell_area=50,
    max_cell_area=1000,
)
print(count_cfg.to_dict())


## Synthetic smoke test

Cell-area thresholds are in pixels² at target magnification. Changing from
20x to 10x changes apparent component area, so revalidate
`min_cell_area`/`max_cell_area` rather than copying them blindly.


In [ ]:
from dataclasses import replace

from rocqipath.analysis import PositiveCellCounter

if RUN_SYNTHETIC_DEMO:
    demo_counter = PositiveCellCounter(
        replace(count_cfg, source_magnification=20.0)
    )
    demo_result = demo_counter.count_slide(
        str(synthetic_slide),
        label="Synthetic DAB",
    )
    print(demo_result)
else:
    print("Set RUN_SYNTHETIC_DEMO=True after installing .[cellcount].")


## Count one real IHC slide

The returned density denominator uses the per-pixel tissue mask, not the
entire rectangular area of accepted tiles. MPP is read from slide metadata;
when absent, the current implementation uses its documented Aperio fallback.
For quantitative work, verify scanner MPP metadata independently.


In [ ]:
if RUN_SINGLE_SLIDE:
    if not IHC_SLIDE.is_file():
        raise FileNotFoundError(IHC_SLIDE)
    counter = PositiveCellCounter(count_cfg)
    single_result = counter.count_slide(str(IHC_SLIDE), label="CD8")
    print(single_result)
else:
    single_result = None
    print("Set RUN_SINGLE_SLIDE=True after editing IHC_SLIDE.")


## Batch counting

Batch discovery is non-recursive. Each successful slide writes an
individual JSON file; the module directory also receives a batch summary.


In [ ]:
if RUN_BATCH:
    if not IHC_BATCH_DIR.is_dir():
        raise FileNotFoundError(IHC_BATCH_DIR)
    counter = PositiveCellCounter(count_cfg)
    batch_results = counter.count_batch(str(IHC_BATCH_DIR), label="CD8")
    print(f"Successful slides: {len(batch_results)}")
else:
    batch_results = []
    print("Set RUN_BATCH=True after checking IHC_BATCH_DIR.")


## Compare ground-truth and predicted IHC slides

Both slides must have identical dimensions at target magnification.
Per-patch Otsu thresholds are computed independently. The workflow writes
JSON, Excel patch results, and optional comparison panels.


In [ ]:
GT_SLIDE = DATA_ROOT / "wsi" / "Sample_0001_cd8_gt.svs"
PRED_SLIDE = DATA_ROOT / "wsi" / "Sample_0001_cd8_pred.tif"

if RUN_PAIRED_COMPARISON:
    if not GT_SLIDE.is_file() or not PRED_SLIDE.is_file():
        raise FileNotFoundError("Edit GT_SLIDE and PRED_SLIDE.")
    paired_cfg = replace(
        count_cfg,
        source_magnification=None,
        paired_source_magnification=20.0,
    )
    counter = PositiveCellCounter(paired_cfg)
    paired_result = counter.count_slide_pair(
        gt_path=str(GT_SLIDE),
        pred_path=str(PRED_SLIDE),
        label="CD8",
        save_plots=True,
        max_plots=10,
        dpi=300,
    )
    print(paired_result)
else:
    paired_result = None
    print("Set RUN_PAIRED_COMPARISON=True after aligning both slides.")


In [ ]:
import json

count_root = OUTPUT_ROOT / "cell_counting"
result_jsons = (
    sorted(count_root.rglob("*_cell_count_results.json"))
    if count_root.is_dir()
    else []
)

print(f"Per-slide JSON results: {len(result_jsons)}")
if result_jsons:
    result_path = result_jsons[0]
    result = json.loads(result_path.read_text(encoding="utf-8"))
    print(json.dumps(result, indent=2))


## Validation checklist

- Review brown-gate and binary masks on weak, strong, necrotic, folded, and
  background-rich regions.
- Confirm one connected component corresponds approximately to one cell,
  not a cluster or fragmented chromogen.
- Validate area thresholds separately at every target magnification.
- Confirm MPP before interpreting cells/mm².
- Report the fixed HSV gate, Otsu strategy, tissue gate, area thresholds,
  target magnification, and scanner cohort in methods.
